# **Поиск вакансий по резюме. Сравнение**

In [ ]:
# Установка библиотеки (если ещё не установлена)
#!pip install python-docx

In [ ]:
#pip install --upgrade transformers

In [27]:
PINECONE_API_KEY = "pcsk_2ghPWB_3v4p9rrdZ85hp2SzJnMMEPELbpE1cPiXhXUaiBQueTq7C3Wy1ag271dGJnbbdG7" # акк Анны

vectorizing_model = 'all-MiniLM-L6-v2'
#vectorizing_model = 'sentence-transformers/distilbert-base-nli-mean-tokens'

index_name = 'res-vac-match-ngramm'
index_dimension=384


# **Часть первая - Наполнение векторной базы**

# Извлечение вакансий из таблицы

In [28]:
from ngramm import get_ngramm, text_preproc
my_ngramm = 3

In [29]:
import pandas as pd
import os

# Путь к папке с датасетом
DATASET_PATH = "vacancy-resume-matching-dataset-main"
csv_file = os.path.join(DATASET_PATH, "5_vacancies.csv")

if not os.path.exists(csv_file):
    raise FileNotFoundError(f"Файл не найден: {csv_file}\nУбедитесь, что папка 'vacancy-resume-matching-dataset-main' находится в текущей директории.")

# Загружаем CSV
df_vacancies = pd.read_csv(csv_file)

# Извлекаем нужные поля и удаляем пустые строки
df_vacancies = df_vacancies[['job_title', 'job_description']].dropna()

# Создаём список chunk_text: каждый элемент — один текст вакансии
chunk_text = []
metadata = []

for idx, row in df_vacancies.iterrows():
    title = row['job_title']
    desc = row['job_description']
    
    # Формируем единый текст для вакансии (один чанк = одна вакансия)
    full_text = f"Job title: {title}. Job description: {desc}"
    
    # Токенируем единый текст для вакансии
    full_text = get_ngramm(my_ngramm,text_preproc(full_text))
    chunk_text.append(full_text)
    metadata.append({
        "job_title": title,
        "job_description": desc,
        "vacancy_id": idx+1
    })

# Выводим статистику
print(f"Создано {len(chunk_text)} чанков (по одному на вакансию) из {len(df_vacancies)} вакансий.")

# Пример первого чанка
print("\nПример первого чанка:")
print(chunk_text[0])
print("Метаданные:")
print(metadata[0])


Создано 5 чанков (по одному на вакансию) из 5 вакансий.

Пример первого чанка:
job tit itl tle sof oft ftw twa war are dev eve vel elo lop ope per net job des esc scr cri rip ipt pti tio ion pri rim ima mar ary pur urp rpo pos ose dev eve vel elo lop opm pme men ent and mai ain int nte ten ena nan anc nce sof oft ftw twa war are app ppl pli lic ica cat ati tio ion ons bui uil ilt usi sin ing mic icr cro ros oso sof oft sta tac ack ess sse sen ent nti tia ial dut uti tie ies and res esp spo pon ons nsi sib ibi bil ili lit iti tie ies fol oll llo low owi win ing sum umm mma mar ary the ess sse sen ent nti tia ial fun unc nct cti tio ion ons for thi his job oth the her dut uti tie ies may per erf rfo for orm rme med bot oth maj ajo jor and min ino nor whi hic ich are not men ent nti tio ion one ned bel elo low spe pec eci cif ifi fic act cti tiv ivi vit iti tie ies may cha han ang nge fro rom tim ime tim ime des esi sig ign dev eve vel elo lop and tes est sof oft ftw twa war are app ppl p

## Подготовка к загрузке в Pinecone

In [30]:

from sentence_transformers import SentenceTransformer

# Загружаем модель
model = SentenceTransformer(vectorizing_model)

# Генерируем эмбеддинги
embeddings = model.encode(chunk_text).tolist()  # Список списков, длина 768

# Генерируем уникальные ID для векторов
ids = [f"chunk_{i}" for i in range(len(chunk_text))]

# Подготавливаем записи для upsert
my_chunks = list(zip(ids, embeddings, metadata))

print(f"✅ Готово к загрузке: {len(my_chunks)} векторов в Pinecone")


✅ Готово к загрузке: 5 векторов в Pinecone


## Загрузка 5-и чанков в Pinecone

In [31]:
from pinecone import Pinecone, ServerlessSpec

# Инициализация Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)


# Создание индекса, если его нет
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=index_dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

# Подключение к индексу
index = pc.Index(index_name)

# Загружаем все чанки из my_chunks (всего 5 — можно без батчей)
print(f"📤 Загружаем {len(my_chunks)} векторов в Pinecone...")
upsert_response = index.upsert(vectors=my_chunks)

# Проверка результата
print(f"✅ Успешно загружено {upsert_response['upserted_count']} векторов.")


📤 Загружаем 5 векторов в Pinecone...
✅ Успешно загружено 5 векторов.


# **Часть вторая - Поиск вакансии по резюме**

### Извлечение текста из вордовских файлов

In [32]:
from docx import Document
import os

def extract_text_from_docx(folder_path, file_number):
    """
    Извлекает текст из файла {file_number}.docx в указанной папке.
    Обрабатывает как параграфы, так и текст внутри таблиц.

    Args:
        folder_path (str): Путь к папке с файлами (например, r"C:\CV")
        file_number (int): Номер файла (1 до 65)

    Returns:
        str: Извлечённый текст из .docx файла или пустая строка, если файл не найден/ошибка
    """
    file_name = f"{file_number}.docx"
    file_path = os.path.join(folder_path, file_name)

    try:
        doc = Document(file_path)
        full_text = []

        # 1. Извлекаем текст из параграфов
        for paragraph in doc.paragraphs:
            if paragraph.text.strip():  # Только непустые параграфы
                full_text.append(paragraph.text.strip())

        # 2. Извлекаем текст из таблиц
        for table in doc.tables:
            for row in table.rows:
                for cell in row.cells:
                    for paragraph in cell.paragraphs:
                        if paragraph.text.strip():  # Только непустые параграфы в ячейках
                            full_text.append(paragraph.text.strip())

        # Объединяем все тексты с разрывами строк
        return '\n'.join(full_text)

    except FileNotFoundError:
        print(f"⚠️ Файл не найден: {file_path}")
        return ""
    except Exception as e:
        print(f"❌ Ошибка при чтении {file_path}: {e}")
        return ""


### Генерация поискового списка и поиск

In [37]:
import os
import time
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone

# Инициализация модели для векторизации
model = SentenceTransformer(vectorizing_model)

# Инициализация Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(index_name)

# Путь к папке CV
CV_FOLDER_PATH = r"vacancy-resume-matching-dataset-main\CV"

# Проверим, существует ли папка
if not os.path.exists(CV_FOLDER_PATH):
    print(f"❌ Папка не существует: {CV_FOLDER_PATH}")
    print("Убедитесь, что путь указан правильно. Возможно, нужно использовать абсолютный путь.")
else:
    print(f"✅ Папка найдена: {CV_FOLDER_PATH}\n")

    # Собираем тексты всех файлов и ищем топ-5 вакансий для каждого
    search_results = []
    
    for i in range(1, 66):  # от 1 до 65 включительно
        print(f"📄 Обработка файла {i}.docx...")
        
        # Извлекаем текст резюме
        text = extract_text_from_docx(CV_FOLDER_PATH, i)
        # токенизируем текст резюме
        text = get_ngramm(my_ngramm,text_preproc(text))
                
        # ЗАДЕРЖКА
        time.sleep(1)

        # Проверить, что текст файла не пустой
        print(text[:70], "...")
        if text:
            # Векторизуем резюме
            resume_embedding = model.encode([text]).tolist()[0]
            
            # Ищем топ-5 вакансий в Pinecone
            search_response = index.query(
                vector=resume_embedding,
                top_k=5,
                include_metadata=True
            )
            
            # Извлекаем порядковые номера вакансий из metadata (vacancy_id)
            # Вакансии уже отсортированы по релевантности (от самой релевантной к наименее)
            top_5_vacancies = []
            for match in search_response.matches:
                vacancy_id = int(match.metadata["vacancy_id"])  # Преобразуем в целое число
                top_5_vacancies.append(vacancy_id)
            
            search_results.append(top_5_vacancies)
            print(f"   ✅ Найдено вакансий: {top_5_vacancies}")
        else:
            # Если текст не извлекся, добавляем пустой список
            search_results.append([])
            print(f"   ⚠️ Текст не извлечен")

    print(f"\n=== Готово! ===")
    print(f"Обработано резюме: {len([r for r in search_results if r])} из 65")
    


✅ Папка найдена: vacancy-resume-matching-dataset-main\CV

📄 Обработка файла 1.docx...
jav ava ful ull sta tac ack dev eve vel elo lop ope per jav ava bac ac ...
   ✅ Найдено вакансий: [1, 2, 3, 5, 4]
📄 Обработка файла 2.docx...
ful ull lls lst sta tac ack dev eve vel elo lop ope per yea ear ars ex ...
   ✅ Найдено вакансий: [2, 5, 4, 1, 3]
📄 Обработка файла 3.docx...
yea ear ars tot ota tal exp xpe per eri rie ien enc nce dev eve vel el ...
   ✅ Найдено вакансий: [2, 5, 1, 3, 4]
📄 Обработка файла 4.docx...
jav ava bac ack cke ken end dev eve vel elo lop ope per wit ith yea ea ...
   ✅ Найдено вакансий: [5, 2, 3, 4, 1]
📄 Обработка файла 5.docx...
yea ear ars pro rov ove ven exp xpe per eri rie ien enc nce yea ear ar ...
   ✅ Найдено вакансий: [2, 5, 4, 3, 1]
📄 Обработка файла 6.docx...
leo eon oni nid sha hal ala lae aev ful ull sta tac ack dev eve vel el ...
   ✅ Найдено вакансий: [4, 5, 2, 1, 3]
📄 Обработка файла 7.docx...
fro ron ont nte ten end dev eve vel elo lop ope per yea ear ar

In [39]:
# Выводим структуру результатов (аналогично ANNOTATOR_1_RANKINGS)
print(f"\n📊 Результаты поиска с внешней токенизацией по {my_ngramm}-граммам (модель - {vectorizing_model}, размер - {index_dimension}):")
print("SEARCH_RESULTS = [\n", end="")
for i in range(0, len(search_results), 5):
    group = search_results[i:i+5]
    
    # Формируем комментарий
    start_num = i + 1
    end_num = min(i + 5, len(search_results))
    comment = f"#{start_num}-{end_num}" if end_num - start_num + 1 == 5 else f"#{start_num}-{end_num}"
    
    # Выводим первую строку группы с комментарием
    if i == 0:
        print(f"{group[0]},{group[1]},{group[2]},{group[3]},{group[4]}, {comment}")
    else:
        print(f"{group[0]},{group[1]},{group[2]},{group[3]},{group[4]}, {comment}")
print("]")



📊 Результаты поиска с внешней токенизацией по 3-граммам (модель - all-MiniLM-L6-v2, размер - 384):
SEARCH_RESULTS = [
[1, 2, 3, 5, 4],[2, 5, 4, 1, 3],[2, 5, 1, 3, 4],[5, 2, 3, 4, 1],[2, 5, 4, 3, 1], #1-5
[4, 5, 2, 1, 3],[5, 2, 4, 1, 3],[2, 5, 4, 3, 1],[2, 4, 5, 1, 3],[5, 3, 2, 1, 4], #6-10
[5, 3, 4, 2, 1],[4, 1, 2, 5, 3],[2, 4, 5, 3, 1],[2, 5, 3, 4, 1],[2, 4, 5, 1, 3], #11-15
[2, 5, 3, 1, 4],[5, 2, 3, 4, 1],[2, 3, 5, 1, 4],[2, 5, 4, 3, 1],[5, 2, 4, 3, 1], #16-20
[5, 2, 3, 4, 1],[2, 5, 3, 4, 1],[2, 4, 3, 5, 1],[4, 1, 5, 3, 2],[4, 2, 5, 3, 1], #21-25
[3, 2, 5, 1, 4],[2, 5, 4, 1, 3],[4, 2, 3, 1, 5],[2, 5, 4, 3, 1],[5, 2, 4, 3, 1], #26-30
[2, 5, 4, 1, 3],[5, 2, 4, 1, 3],[5, 2, 4, 1, 3],[5, 2, 1, 4, 3],[2, 4, 3, 5, 1], #31-35
[5, 4, 1, 2, 3],[5, 2, 4, 1, 3],[5, 2, 1, 3, 4],[5, 2, 1, 4, 3],[2, 5, 4, 3, 1], #36-40
[2, 5, 4, 1, 3],[3, 2, 1, 5, 4],[2, 4, 5, 3, 1],[2, 5, 4, 3, 1],[5, 2, 4, 1, 3], #41-45
[4, 2, 5, 3, 1],[5, 2, 4, 1, 3],[2, 5, 3, 4, 1],[5, 2, 1, 3, 4],[3, 1, 2, 5, 4], #46-50
[3, 

### 🔥 Удаление индекса 

In [40]:
# Удалить индекс через клиент
pc.delete_index(index_name)

print(f"✅ Индекс '{index_name}' успешно удалён.")


✅ Индекс 'res-vac-match-ngramm' успешно удалён.
